In [1]:
import os
import time
import torch
import numpy as np
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import WikipediaNetwork
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [2]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Net2(torch.nn.Module):
    def __init__(self, num_features, hidden, num_layers):
        super(Net2, self).__init__()
        self.num_layers = num_layers
        self.conv = torch.nn.ModuleList()
        self.conv.append(GCNConv(num_features, hidden))
        for i in range(self.num_layers - 1):
            self.conv.append(GCNConv(hidden, hidden))
        self.lt1 = torch.nn.Linear(hidden, 1)

    def reset_parameters(self):
        for module in self.conv:
            module.reset_parameters()
        self.lt1.reset_parameters()

    def forward(self, x, edge_index):
        for i in range(self.num_layers):
            x = self.conv[i](x, edge_index)
            x = F.elu(x)
            x = F.dropout(x, training=self.training)
        x = self.lt1(x)
        return x

def index_to_mask(index, size):
    mask = torch.zeros(size, dtype=torch.bool, device=index.device)
    mask[index] = 1
    return mask

def splits_regression(data, train_ratio, val_ratio):
    if train_ratio + val_ratio >= 1:
        raise ValueError('train_ratio + val_ratio should be less than 1')
    num_nodes = data.x.shape[0]
    num_train = int(train_ratio*num_nodes)
    num_val = int(val_ratio*num_nodes)
    perm_nodes = torch.randperm(num_nodes)
    train_nodes = perm_nodes[:num_train]
    val_nodes = perm_nodes[num_train:num_train+num_val]
    test_nodes = perm_nodes[num_train+num_val:]
    data.train_mask = index_to_mask(train_nodes, size=num_nodes)
    data.val_mask = index_to_mask(val_nodes, size=num_nodes)
    data.test_mask = index_to_mask(test_nodes, size=num_nodes)

    return data

In [3]:
if not os.path.exists("../save"):
  os.mkdir("../save")
if not os.path.exists("../save/node_reg"):
  os.mkdir("../save/node_reg")
if not os.path.exists("../save/node_reg/baselines"):
  os.mkdir("../save/node_reg/baselines")

dataset_names = ['chameleon']
for dataset_name in dataset_names:
    dataset = WikipediaNetwork(root='../dataset', name=dataset_name, geom_gcn_preprocess=False)
    data = splits_regression(dataset[0], 0.3, 0.2)
    model = Net2(data.x.shape[1], 512, 2).to(device)
    loss_fn = torch.nn.L1Loss()
    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    all_loss = []
    avg_time = 0
    for run in range(20):
        best_val_loss = 100000
        model.reset_parameters()
        data = data.to(device)
        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            out = model(data.x, data.edge_index)
            loss = loss_fn(out[data.train_mask].view(-1, 1), data.y[data.train_mask].view(-1, 1))
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                out = model(data.x, data.edge_index)
                val_loss = loss_fn(out[data.val_mask].view(-1, 1), data.y[data.val_mask].view(-1, 1))
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    #save model
                    torch.save(model.state_dict(), f'../save/node_reg/baselines/baseline_{dataset_name}.pt')
        start = time.time()
        out = model(data.x, data.edge_index)
        avg_time += time.time() - start
        test_loss = loss_fn(out[data.test_mask].view(-1, 1), data.y[data.test_mask].view(-1, 1))
        all_loss.append(test_loss.item()/data.y[data.test_mask].std().item())

    top_loss = sorted(all_loss)[:10]
    print(f'{dataset_name} fixed: {np.mean(all_loss):.4f} ± {np.std(all_loss):.4f} time: {avg_time/20:.4f}')

chameleon fixed: 0.5234 ± 0.0141 time: 0.0025
